# Srota zero-shot baseline on OpenSLR 104 test

This notebook runs [`Surajgameramp/srota`](https://huggingface.co/Surajgameramp/srota) zero-shot on the OpenSLR 104 Hinglish test set (2,942 clips, 4.76 h) and scores it with the project normalizer and metrics in `src/hinglish_asr/`. It runs in Google Colab on an L4 GPU.

Srota is Qwen3-ASR-0.6B fully fine-tuned on about 95 h of Hinglish: HiACC conversational speech plus OpenSLR 104 tutorial speech. It is a reference point (model 4 in the README) for what a small Hinglish fine-tune reaches. "Zero-shot" here means this project does no further training; Srota has already seen OpenSLR 104 training data, so its score on this test set is in-domain.

This notebook runs the same code as the Qwen3-ASR-0.6B notebook, so the two scores differ only by the model weights.

- **Inputs:** the project folder on Google Drive, and `openslr104_test.zip` (the clips in `data/processed/openslr104/test/`).
- **Decoding:** `qwen-asr` 0.0.6 defaults: greedy, up to 512 new tokens, and the package's own repetition clean-up on each output. `language=None`, as Srota's card requires (Srota was trained with that prompt). bf16 with the default attention, not FlashAttention 2. Whole clips, as in Srota's own evaluation and the Qwen3-ASR notebook.
- **Outputs:** one row in `results/summary.csv`, and each clip's output and detected language in `results/srota-zeroshot/hyps.jsonl`.

Note: with the current scoring code T-WER equals WER, because `Normalizer.normalize` converts Devanagari loanwords to English before `t_wer` runs.

## Setup

`qwen-asr` 0.0.6 pins `transformers==4.57.6`, older than Colab's version, so it is installed before anything imports `transformers`. If Colab asks to restart the session after the install, restart and continue from the next cell.

In [1]:
!pip install -q qwen-asr==0.0.6 jiwer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/projects/asr"
ZIP_PATH = f"{PROJECT}/openslr104_test.zip"
AUDIO_DIR = "/content/openslr104_test"

Mounted at /content/drive


In [3]:
import os
import sys

import torch

os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")
print(torch.cuda.get_device_name(0))

NVIDIA L4


In [4]:
import zipfile

from tqdm.auto import tqdm

with zipfile.ZipFile(ZIP_PATH) as z:
    for member in tqdm(z.infolist(), desc="unzip"):
        z.extract(member, AUDIO_DIR)

unzip:   0%|          | 0/5886 [00:00<?, ?it/s]

## Test manifest

The manifest's `audio_path` values are local paths on the machine that made the data, so each clip is found by file name in the unzipped folder.

In [5]:
import json
from pathlib import Path

wavs = {p.name: str(p) for p in Path(AUDIO_DIR).rglob("*.wav")}
with open("data/processed/openslr104/test_manifest.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

missing = [r["utt_id"] for r in rows if Path(r["audio_path"]).name not in wavs]
assert not missing, f"{len(missing)} clips missing, e.g. {missing[:3]}"

paths = [wavs[Path(r["audio_path"]).name] for r in rows]
refs = [r["text"] for r in rows]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} clips in manifest, {len(wavs)} wav files, {hours:.2f} h")
print(refs[0])
print(paths[0])

2942 clips in manifest, 5884 wav files, 4.76 h
libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
/content/openslr104_test/test/598753_406yMKxIdSDHRf8H_0000.wav


## Model

In [6]:
from qwen_asr import Qwen3ASRModel

MODEL = "Surajgameramp/srota"
BATCH = 32

model = Qwen3ASRModel.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    max_inference_batch_size=BATCH,
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

A check on 5 clips before the full run. The detected language is shown in brackets.

In [7]:
for ref, res in zip(refs[:5], model.transcribe(audio=paths[:5], language=None)):
    print("REF:", ref)
    print(f"HYP ({res.language}):", res.text, "\n")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP (): लिबर ऑफिस impress पर slide master और slide design इस spoken tutorial में आपका स्वागत है 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP (): इस tutorial में हम सीखेंगे कि slides के लिए backgrounds layouts कैसे लागू करें 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP (): background slide पर लागू किए गए सभी रंगों और प्रभावों को उज़्वित करता है जो विषय वस्तु के पीछे मौजूद हैं 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP (): libreoffice impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP (): आप भी अपने खुद के custom backgrounds बना सकते हैं 



## Transcribe the test set

The clips go to `transcribe` one batch at a time, so tqdm can show progress.

In [8]:
import time
from collections import Counter

results = []
start = time.time()
for b in tqdm(range(0, len(paths), BATCH), desc="transcribe"):
    results += model.transcribe(audio=paths[b:b + BATCH], language=None)
elapsed = time.time() - start

hyps = [r.text for r in results]
assert len(hyps) == len(rows)
print(f"{len(hyps)} outputs, {sum(not h.strip() for h in hyps)} empty")
print(f"{elapsed / 60:.1f} min, real-time factor {elapsed / (hours * 3600):.4f}")
print("detected languages:", Counter(r.language for r in results).most_common())

transcribe:   0%|          | 0/92 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

2942 outputs, 0 empty
15.2 min, real-time factor 0.0533
detected languages: [('', 2942)]


Each clip's output is saved before scoring, so the clips can be re-scored later without the GPU.

In [9]:
out_dir = Path("results/srota-zeroshot")
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "hyps.jsonl", "w", encoding="utf-8") as f:
    for r, res in zip(rows, results):
        f.write(json.dumps({"utt_id": r["utt_id"], "ref": r["text"], "hyp": res.text, "language": res.language}, ensure_ascii=False) + "\n")

## Score

In [10]:
from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

norm = Normalizer("data/lists/loanwords.csv")
assert score(refs, refs, norm) == {"wer": 0.0, "t_wer": 0.0}

scores = score(refs, hyps, norm)
print(f"WER {scores['wer']:.4f}  T-WER {scores['t_wer']:.4f}")

WER 0.3059  T-WER 0.3059


Normalized text that the metrics compare, for the first 5 clips:

In [11]:
for ref, hyp in zip(refs[:5], hyps[:5]):
    print("REF:", norm.for_scoring(ref))
    print("HYP:", norm.for_scoring(hyp), "\n")

REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP: libre office impress पर slide master और slide design इस spoken tutorial में आपका स्वागत है 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP: इस tutorial में हम सीखेंगे कि slides के लिए backgrounds layouts कैसे लागू करें 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP: background slide पर लागू किए गए सभी रंगों और प्रभावों को उज़्वित करता है जो विषय वस्तु के पीछे मौजूद हैं 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP: libreoffice impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP: आप भी अपने खुद के custom backgrounds बना सकते हैं 



## Results file

In [12]:
import csv
import datetime

summary = Path("results/summary.csv")
row = {
    "model": MODEL,
    "dataset": "openslr104",
    "split": "test",
    "n_utts": len(rows),
    "wer": round(scores["wer"], 4),
    "t_wer": round(scores["t_wer"], 4),
    "decoding": "zero-shot, qwen-asr 0.0.6 defaults (greedy, max 512 new tokens, repetition clean-up), language=None, bf16, whole clips",
    "date": datetime.date.today().isoformat(),
}
write_header = not summary.exists()
with open(summary, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print(summary.read_text(encoding="utf-8"))

model,dataset,split,n_utts,wer,t_wer,decoding,date
openai/whisper-small,openslr104,test,2942,0.915,0.915,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi, fp16, 30 s chunks",2026-11-23
Trelis/tara,openslr104,test,2942,0.295,0.295,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi + mixedcode, bf16, 30 s pieces",2026-11-23
Qwen/Qwen3-ASR-0.6B,openslr104,test,2942,0.4362,0.4362,"zero-shot, qwen-asr 0.0.6 defaults (greedy, max 512 new tokens, repetition clean-up), language=None, bf16, whole clips",2026-11-23
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4584,0.4584,"zero-shot, greedy RNN-T, streaming latency 560 ms, language=auto, float32, whole clips",2026-11-23
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4546,0.4546,"zero-shot, greedy RNN-T, streaming latency 1120 ms, language=auto, float32, whole clips",2026-11-23
Surajgameramp/srota,openslr104,test,29